In [1]:
%cd /kaggle/working
!git clone https://github.com/shujaabbasn/ATML.git
%cd /kaggle/working/ATML/pa4
!python -m pip install -q -r requirements.txt
# kaggle ships torchao 0.10, older peft refuses to build lora while it is installed, nothing here uses torchao
!python -m pip uninstall -y -q torchao
!python -c "from peft.import_utils import is_torchao_available; print('torchao used by peft:', is_torchao_available())"
!nvidia-smi

/kaggle/working
Cloning into 'ATML'...
remote: Enumerating objects: 1218, done.
remote: Counting objects: 100% (666/666), done.
remote: Compressing objects: 100% (428/428), done.
remote: Total 1218 (delta 276), reused 620 (delta 234), pack-reused 552 (from 1)
Receiving objects: 100% (1218/1218), 36.20 MiB | 35.44 MiB/s, done.
Resolving deltas: 100% (506/506), done.
/kaggle/working/ATML/pa4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompati

In [2]:
# staff data repo (scripts + bundled files), then merge anything attached from earlier sessions
!git clone -q https://github.com/AbDu11aHHH/atml-pa4-datasets.git data_repo
!python kaggle_runs/attach_inputs.py
# skips files that already exist, so with data_prepare attached this is only a check (C4 streaming is slow otherwise)
!cd data_repo && python scripts/setup_data.py --public && python scripts/check_setup.py

unzip /kaggle/input/notebooks/shujabbasn/data-prepare-run/data_repo.zip
unzip /kaggle/input/notebooks/shujabbasn/task3-sampler-baseline-runn/task3_baseline_results.zip
copy /kaggle/input/notebooks/shujabbasn/data-prepare-run/ATML/pa4/data_repo/data -> data_repo/data
copy /kaggle/input/notebooks/shujabbasn/task3-sampler-baseline-runn/ATML/pa4/data_repo/data -> data_repo/data
copy /kaggle/input/notebooks/shujabbasn/task3-sampler-baseline-runn/ATML/pa4/results -> results
merged 5 inputs
revision lock present
tokenizer_config.json: 7.23kB [00:00, 35.8MB/s]
vocab.json: 2.78MB [00:00, 133MB/s]
merges.txt: 1.67MB [00:00, 147MB/s]
tokenizer.json: 7.03MB [00:00, 202MB/s]
Public data available locally. Preserve data/local_source_revisions.json for reproducibility.
OK C4 calibration 128
OK C4 development 32
OK C4 validation windows 177
OK WikiText-2 test windows 437
OK HellaSwag examples 2000
OK GSM8K train 500
OK GSM8K test 60
OK prefix banks
OK source snapshot lock
PASS: public assets prepared 

In [3]:
# needs task3_sampler_baseline_run output attached (teacher bank in results/task3/distill)
!ls results/task3/distill

teacher_bank_cost.json	teacher_train_responses.jsonl.gz


In [4]:
!python -m task3_speculative.distill --step train --method off_policy --seed 6304

I0000 00:00:1795560760.577989     138 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
tokenizer_config.json: 7.30kB [00:00, 25.3MB/s]
vocab.json: 2.78MB [00:00, 104MB/s]
merges.txt: 1.67MB [00:00, 153MB/s]
tokenizer.json: 7.03MB [00:00, 207MB/s]
tokenizer_config.json: 7.30kB [00:00, 31.7MB/s]
vocab.json: 2.78MB [00:00, 155MB/s]
merges.txt: 1.67MB [00:00, 150MB/s]
tokenizer.json: 7.03MB [00:00, 207MB/s]
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 7.75MB/s]
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:11<00:00, 260MB/s]
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 4.66MB/s]
model.safetensors: 100%|██████████████████████| 988M/988M [00:01<00:00, 545MB/s]
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 2.26MB/s]
update 0 kl 0.1085
update 20 kl 0.0691
update 40 kl 0.2298
update 60 kl 0.1882
update 80 kl 0.0708
update 100 kl 0.1288
update 120 kl 0.1005
update 140

In [5]:
!python -m task3_speculative.distill --step train --method off_policy --seed 6305

I0000 00:00:1795561004.457756     222 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
update 0 kl 0.13
update 20 kl 0.1222
update 40 kl 0.0764
update 60 kl 0.1917
update 80 kl 0.1358
update 100 kl 0.0442
update 120 kl 0.1018
update 140 kl 0.1539
merged off_policy_s6305 [{'max_abs_logit_diff': 0.16015625, 'argmax_agree': 1.0}, {'max_abs_logit_diff': 0.171875, 'argmax_agree': 1.0}, {'max_abs_logit_diff': 0.1875, 'argmax_agree': 0.9799999594688416}, {'max_abs_logit_diff': 0.15234375, 'argmax_agree': 1.0}]


In [6]:
!zip -r /kaggle/working/task3_off_results.zip results/task3/drafts

  adding: results/task3/drafts/ (stored 0%)
  adding: results/task3/drafts/off_policy_s6304/ (stored 0%)
  adding: results/task3/drafts/off_policy_s6304/merge_check.json (deflated 66%)
  adding: results/task3/drafts/off_policy_s6304/adapter/ (stored 0%)
  adding: results/task3/drafts/off_policy_s6304/adapter/adapter_config.json (deflated 60%)
  adding: results/task3/drafts/off_policy_s6304/adapter/adapter_model.safetensors (deflated 7%)
  adding: results/task3/drafts/off_policy_s6304/adapter/README.md (deflated 65%)
  adding: results/task3/drafts/off_policy_s6304/cost.json (deflated 58%)
  adding: results/task3/drafts/off_policy_s6304/presentations.jsonl (deflated 86%)
  adding: results/task3/drafts/off_policy_s6304/merged/ (stored 0%)
  adding: results/task3/drafts/off_policy_s6304/merged/merges.txt (deflated 57%)
  adding: results/task3/drafts/off_policy_s6304/merged/generation_config.json (deflated 39%)
  adding: results/task3/drafts/off_policy_s6304/merged/chat_template.jinja (defl